# Lab 01 — Part F: Experiment 2 — Preprocessing ablation

Notebook thực hiện mục 9.1–9.5 của đề Lab 01 trên cùng corpus và chuẩn TF-IDF đã dùng ở Chapter 7. Các pipeline được fit trên 30K document, sau đó so sánh representation và search evidence.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

WORKING_DIRECTORY = Path.cwd().resolve()
CANDIDATE_EXERCISE_DIRECTORIES = (
    WORKING_DIRECTORY,
    WORKING_DIRECTORY / "Chapter8",
    WORKING_DIRECTORY / "lab1" / "Chapter8",
)
EXERCISE_DIRECTORY = next(
    path
    for path in CANDIDATE_EXERCISE_DIRECTORIES
    if (path / "run_experiments.py").is_file()
)
if str(EXERCISE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(EXERCISE_DIRECTORY))

from run_experiments import (
    ABLATION_RELEVANCE_LABELS,
    DATASET_PATH,
    OUTPUT_PATH,
    QUERIES,
    REPORT_PATH,
    TOP_DOCUMENT_COUNT,
    build_vectorizers,
    calculate_query_oov,
    fit_pipelines,
    load_documents,
    matrix_metrics,
    mean_precision_at_k,
    write_candidates,
    write_report,
)

documents = load_documents(DATASET_PATH)
vectorizers = build_vectorizers()
print(f"Documents loaded: {len(documents):,}")


In [ ]:
# 9.1 — Pipeline A (Minimal): lowercase → word tokenization → TF-IDF.
pipeline_a = vectorizers["A_minimal"]

print("Pipeline A: lowercase + word tokenization")
print(f"Token pattern: {pipeline_a.token_pattern}")


In [ ]:
# 9.2 — Pipeline B (Normalized): lowercase → accent normalization → tokenization → stopword handling → TF-IDF.
pipeline_b = vectorizers["B_normalized"]

print("Pipeline B: lowercase + Unicode accent normalization + word tokenization + English stopword removal")
print(f"Token pattern: {pipeline_b.token_pattern}")


In [ ]:
# 9.3 — Pipeline C (Extended): normalization → character subword tokenization → TF-IDF.
pipeline_c = vectorizers["C_subword"]

print("Pipeline C: lowercase + Unicode accent normalization + character 3–5 grams")
print(f"N-gram range: {pipeline_c.ngram_range}; min_df: {pipeline_c.min_df}; max_features: {pipeline_c.max_features:,}")


In [ ]:
# 9.4 — So sánh vocabulary size, average features/document, sparsity, OOV rate và Mean P@5.
results = fit_pipelines(documents)
metrics = {
    name: matrix_metrics(result, len(documents))
    for name, result in results.items()
}
mean_oov = {
    name: sum(calculate_query_oov(result, query) for query in QUERIES) / len(QUERIES)
    for name, result in results.items()
}
mean_precision = {
    name: mean_precision_at_k(result, ABLATION_RELEVANCE_LABELS, TOP_DOCUMENT_COUNT)
    for name, result in results.items()
}
comparison_rows = tuple(
    f"{name}: vocabulary={metrics[name]['vocabulary_size']:,}; "
    f"avg_features/document={metrics[name]['average_nonzero_features']:.2f}; "
    f"sparsity={metrics[name]['sparsity']:.6%}; "
    f"mean_OOV={mean_oov[name]:.2%}; Mean_P@{TOP_DOCUMENT_COUNT}={mean_precision[name]:.3f}"
    for name in results
)

print("\n".join(comparison_rows))


In [ ]:
# 9.5 — Lưu Top-K candidates, report số liệu và diễn giải dựa trên evidence.
from IPython.display import Markdown, display

write_candidates(documents, results, OUTPUT_PATH)
write_report(documents, results, REPORT_PATH)
display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))


## Câu hỏi phân tích

Report ở mục 9.5 trả lời các câu hỏi bắt buộc: tác động của lowercasing, stopword removal, punctuation; pipeline nào sparse nhất; pipeline nào có Mean P@5 cao nhất trên tập nhãn nhỏ; và vì sao vocabulary nhỏ hơn không tự động làm search tốt hơn. Kết luận phải dựa trên bảng số liệu vừa chạy.